In [ ]:
from snowflake.snowpark.context import get_active_session
from sklearn import datasets
import tempfile,shutil, os
import pandas as pd

import time, math
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

from snowflake.ml.modeling.ensemble.random_forest_regressor import RandomForestRegressor as RFG
from snowflake.ml.modeling.model_selection.grid_search_cv import GridSearchCV as GSC

In [ ]:
session=get_active_session()

In [ ]:
session

In [ ]:
df=pd.read_csv(filepath_or_buffer="housing.csv")

In [ ]:
df.head()

In [ ]:
df.columns=[c.upper() for c in df.columns]

In [ ]:
df.head()

In [ ]:
df=session.create_dataframe(data=df)

In [ ]:
df.show()

In [ ]:
df.write.mode(save_mode="overwrite").save_as_table(table_name="test.public.california_housing")

In [ ]:
df.count()

## Doing Hyperparameter Tunings with sckitlearn

In [ ]:
df=session.table(name="test.public.california_housing").to_pandas()

In [ ]:
df.head()

In [ ]:
list(df.columns)

In [ ]:
FEATS=["LONGITUDE","LATITUDE","HOUSING_MEDIAN_AGE","TOTAL_ROOMS","TOTAL_BEDROOMS",
      "POPULATION","POPULATION","MEDIAN_INCOME","OCEAN_PROXIMITY"]
LABEL=["MEDIAN_HOUSE_VALUE"]

In [ ]:
X,y=df.loc[:,FEATS], df.loc[:,LABEL]

In [ ]:
y

In [ ]:
# y['OCEAN_PROXIMITY'].unique()
mapped_dict={y.unique()[i]:i for i in range(len(y.unique()))}
mapped_dict_rev={j:i for i,j in mapped_dict.items()}
mapped_dict

In [ ]:
y=y.apply(lambda k: mapped_dict[k])
y

In [ ]:
pipe=GridSearchCV(
    estimator=RandomForestRegressor(),
    param_grid={
        "max_depth":[80,90,100,110],
        "min_samples_leaf":[1,3,10],
        "min_samples_split":[2,3,10],
        "n_estimators":[100,200,400]
    },
    cv=5
)

In [ ]:
start=time.time()
pipe.fit(X=X,y=y)

end=time.time()
print(f"Total time taken: {end-start} seconds")

In [ ]:
pipe.best_params_

In [ ]:
pipe.best_score

## Doing Hyperparameter Tunings with snowflake

In [ ]:
df=session.table(name="test.public.california_housing")

In [ ]:
df.show()

In [ ]:
session

In [ ]:
pipeSF=GSC(
    estimator=RFG(),
    param_grid={
        "max_depth":[80,90,100,110],
        "min_samples_leaf":[1,3,10],
        "min_samples_split":[2,3,10],
        "n_estimators":[100,200,400]
    },
    cv=5,
    input_cols=[c for c in df.columns if not c.startswith("MEDIAN_HOUSE_VALUE")],
    label_cols=['MEDIAN_HOUSE_VALUE']
)

In [ ]:
[c for c in df.columns if not c.startswith("MEDHOUSEVAL")]

In [ ]:
df.columns

In [ ]:
start=time.time()
pipeSF.fit(dataset=df)
end=time.time()
print(f"Total time taken: {end-start} seconds")

In [ ]:
skpipe=pipeSF.to_sklearn()

In [ ]:
skpipe.best_params_

In [ ]:
skpipe.best_score